<a href="https://colab.research.google.com/github/RainanCoelho/Projeto-Analise-Exploratoria-Grupo8-Gestao_de_Vendedores-Parceiros/blob/main/Pipeline_preprocessamentofinal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/RainanCoelho/Projeto-Analise-Exploratoria-Grupo8-Gestao_de_Vendedores-Parceiros/blob/main/Pipeline_preprocessamento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**FRENTE 3: PIPELINE DE PRÉ-PROCESSAMENTO**

A função `preprocessar(orders, items, products, reviews)` recebe os CSVs brutos da Olist e devolve a tabela analítica, com uma linha por item de pedido. Ela reúne o tratamento das Frentes 1 e 2 e mantém só as colunas usadas nas perguntas do briefing:

1. Quais vendedores mais atrasam o despacho?
2. Vendedores que atrasam recebem avaliações piores?
3. Que critérios objetivos usar para acompanhar os vendedores?

PARTE 1: BIBLIOTECAS E CARGA

In [ ]:
# [IA] registro 2
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, StandardScaler

BASE = "https://raw.githubusercontent.com/RainanCoelho/Projeto-Analise-Exploratoria-Grupo8-Gestao_de_Vendedores-Parceiros/refs/heads/main/"

ARQUIVOS = {
    "orders": "olist_orders_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "products": "olist_products_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
}


def carregar_brutos(base=BASE, arquivos=ARQUIVOS):
    return {nome: pd.read_csv(base.rstrip("/") + "/" + arq) for nome, arq in arquivos.items()}

PARTE 2: LIMPEZA (FRENTE 1)

In [ ]:
# [IA] registro 2
DATAS_ORDERS = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
                "order_delivered_customer_date", "order_estimated_delivery_date"]


def limpar_orders(df):
    df = df.drop_duplicates(subset="order_id").copy()
    for col in DATAS_ORDERS:
        df[col] = pd.to_datetime(df[col], errors="coerce")
    # Despacho antes da compra é impossível: anula a data e mantém o pedido
    erro = df["order_delivered_carrier_date"] < df["order_purchase_timestamp"]
    df.loc[erro, "order_delivered_carrier_date"] = pd.NaT
    return df


def limpar_items(df):
    df = df.drop_duplicates().copy()
    df["shipping_limit_date"] = pd.to_datetime(df["shipping_limit_date"], errors="coerce")
    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    # Remove datas depois de 2018, fora do período do dataset
    return df[df["shipping_limit_date"].dt.year <= 2018]


def limpar_products(df):
    df = df.drop_duplicates(subset="product_id").copy()
    cat = df["product_category_name"]
    df["product_category_name"] = cat.where(cat.isna(), cat.astype(str).str.strip().str.lower()).fillna("sem_categoria")
    return df[["product_id", "product_category_name"]]


def limpar_reviews(df):
    df = df.drop_duplicates().copy()
    df["review_answer_timestamp"] = pd.to_datetime(df["review_answer_timestamp"], errors="coerce")
    df["review_score"] = pd.to_numeric(df["review_score"], errors="coerce")
    df.loc[~df["review_score"].between(1, 5), "review_score"] = np.nan
    # Pedido com mais de uma avaliação: fica a mais recente para não duplicar itens no merge
    df = df.sort_values("review_answer_timestamp").drop_duplicates(subset="order_id", keep="last")
    return df[["order_id", "review_score"]]

PARTE 3: MERGE (base em order_items, left join para não perder itens)

In [ ]:
# [IA] registro 2
def unir_tabelas(items, orders, products, reviews):
    tabela = items.merge(orders, on="order_id", how="left")
    tabela = tabela.merge(products, on="product_id", how="left")
    return tabela.merge(reviews, on="order_id", how="left")

PARTE 4: FEATURE ENGINEERING (FRENTE 2)

In [ ]:
# [IA] registro 2
def dias(fim, inicio):
    return (fim - inicio).dt.total_seconds() / 86400


def criar_features(df):
    out = df.copy()

    # Diferença entre datas
    out["atraso_despacho_dias"] = dias(out["order_delivered_carrier_date"], out["shipping_limit_date"])
    transporte = dias(out["order_delivered_customer_date"], out["order_delivered_carrier_date"])
    out["tempo_transporte_dias"] = transporte.where(transporte >= 0)
    out["atraso_entrega_dias"] = dias(out["order_delivered_customer_date"], out["order_estimated_delivery_date"])

    # Flags binárias (NaN quando falta a data)
    out["flag_despacho_atrasado"] = np.where(out["atraso_despacho_dias"].notna(), (out["atraso_despacho_dias"] > 0).astype(int), np.nan)
    out["flag_entregue_com_atraso"] = np.where(out["atraso_entrega_dias"].notna(), (out["atraso_entrega_dias"] > 0).astype(int), np.nan)
    out["flag_pedido_cancelado"] = out["order_status"].isin(["canceled", "unavailable"]).astype(int)
    out["flag_avaliacao_negativa"] = np.where(out["review_score"].notna(), (out["review_score"] <= 2).astype(int), np.nan)

    # Discretização
    out["faixa_avaliacao"] = pd.cut(out["review_score"], bins=[0, 2, 3, 5], labels=["negativa", "neutra", "positiva"])

    # Agregação por vendedor
    out["qtd_itens_vendedor"] = out.groupby("seller_id")["order_id"].transform("count")
    out["porte_vendedor"] = pd.cut(out["qtd_itens_vendedor"], bins=[0, 10, 100, np.inf], labels=["pequeno", "medio", "grande"])
    return out

PARTE 5: CODIFICAÇÃO E NORMALIZAÇÃO

In [ ]:
# [IA] registro 2
COLUNAS_ESCALAR = ["atraso_despacho_dias", "atraso_entrega_dias", "review_score"]


def codificar_e_escalonar(df):
    out = df.copy()

    # One-hot do porte (categorias fixas, então as colunas são sempre as mesmas)
    out = pd.concat([out, pd.get_dummies(out["porte_vendedor"], prefix="porte", dtype=int)], axis=1)

    # MinMax (0 a 1) e Z-score (média 0, desvio 1)
    if len(out) == 0:
        for col in COLUNAS_ESCALAR:
            out[col + "_minmax"], out[col + "_zscore"] = np.nan, np.nan
        return out
    dados = out[COLUNAS_ESCALAR].astype(float)
    out[[c + "_minmax" for c in COLUNAS_ESCALAR]] = MinMaxScaler().fit_transform(dados)
    out[[c + "_zscore" for c in COLUNAS_ESCALAR]] = StandardScaler().fit_transform(dados)
    return out

PARTE 6: SELEÇÃO DE ATRIBUTOS (colunas usadas nas três perguntas)

In [ ]:
# [IA] registro 2
COLUNAS_FINAIS = [
    "order_id", "order_item_id", "seller_id", "product_id", "product_category_name",
    "order_status", "order_purchase_timestamp", "price",
    "atraso_despacho_dias", "flag_despacho_atrasado",        # pergunta 1
    "review_score", "faixa_avaliacao", "flag_avaliacao_negativa",
    "atraso_entrega_dias", "flag_entregue_com_atraso", "tempo_transporte_dias",  # pergunta 2
    "qtd_itens_vendedor", "porte_vendedor", "porte_pequeno", "porte_medio", "porte_grande",
    "flag_pedido_cancelado",                                 # pergunta 3
] + [c + s for c in COLUNAS_ESCALAR for s in ["_minmax", "_zscore"]]

PARTE 7: FUNÇÃO DO PIPELINE

In [ ]:
# [IA] registro 2
def preprocessar(orders, items, products, reviews):
    tabela = unir_tabelas(limpar_items(items), limpar_orders(orders), limpar_products(products), limpar_reviews(reviews))
    tabela = criar_features(tabela)
    tabela = codificar_e_escalonar(tabela)
    tabela = tabela[COLUNAS_FINAIS].reset_index(drop=True)
    print("Tabela final:", tabela.shape[0], "linhas x", tabela.shape[1], "colunas")
    return tabela

PARTE 8: EXECUÇÃO NA BASE COMPLETA E EXPORTAÇÃO

In [ ]:
# [IA] registro 2
brutos = carregar_brutos()
tabela_analitica = preprocessar(**brutos)
tabela_analitica.to_csv("tabela_analitica.csv", index=False, float_format="%.4f")
tabela_analitica.head()

Tabela final: 112646 linhas x 28 colunas


,order_id,order_item_id,seller_id,product_id,product_category_name,order_status,order_purchase_timestamp,price,atraso_despacho_dias,flag_despacho_atrasado,...,porte_pequeno,porte_medio,porte_grande,flag_pedido_cancelado,atraso_despacho_dias_minmax,atraso_despacho_dias_zscore,atraso_entrega_dias_minmax,atraso_entrega_dias_zscore,review_score_minmax,review_score_zscore
0,00010242fe8c5a6d1ba2dd792cb16214,1,48436dade18ac8b2bce089ec2a041202,4244733e06e7ecb4970a6e2683c13e61,cool_stuff,delivered,2017-09-13 08:59:02,58.90,0.367141,1.0,...,0,0,1,0,0.446273,0.977849,0.411966,0.327043,1.00,0.697026
1,00018f77f2f0320c557190d7a144bdd3,1,dd7ddc04e1b6c2c614352b383efe2d36,e5f2d52b802189ee658865ca93d83a8f,pet_shop,delivered,2017-04-26 10:53:06,239.90,1.145683,1.0,...,0,0,1,0,0.449977,1.185955,0.428924,0.886681,0.75,-0.023705
2,000229ec398224ef6ca0657da4fc703e,1,5b51032eddd242adc84c38acab88f23d,c777355d18b72b67abbeef9df44fd0fd,moveis_decoracao,delivered,2018-01-14 14:33:31,199.00,-2.091458,0.0,...,0,1,0,0,0.434577,0.320658,0.395745,-0.208236,1.00,0.697026
3,00024acbcdf0a6daa1e931b038114c75,1,9d7a1d34a5052409006425275ba1c2b4,7634da152a4610f1595efa32f14722fc,perfumaria,delivered,2018-08-08 10:00:35,12.99,-4.862708,0.0,...,0,1,0,0,0.421394,-0.420105,0.419654,0.580767,0.75,-0.023705
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,df560393f3a51e74553ab94004ba5c87,ac6c3623068f30de03045865e4e10089,ferramentas_jardim,delivered,2017-02-04 13:57:51,199.90,2.825208,1.0,...,0,1,0,0,0.457967,1.634898,0.390196,-0.391354,1.00,0.697026


PARTE 9: TESTE EM UMA AMOSTRA PEQUENA

In [ ]:
# [IA] registro 2
pedidos = brutos["orders"].sample(200, random_state=42)
amostra = {
    "orders": pedidos,
    "items": brutos["items"][brutos["items"]["order_id"].isin(pedidos["order_id"])],
    "products": brutos["products"],
    "reviews": brutos["reviews"][brutos["reviews"]["order_id"].isin(pedidos["order_id"])],
}
tabela_amostra = preprocessar(**amostra)
tabela_amostra.head()

Tabela final: 228 linhas x 28 colunas


,order_id,order_item_id,seller_id,product_id,product_category_name,order_status,order_purchase_timestamp,price,atraso_despacho_dias,flag_despacho_atrasado,...,porte_pequeno,porte_medio,porte_grande,flag_pedido_cancelado,atraso_despacho_dias_minmax,atraso_despacho_dias_zscore,atraso_entrega_dias_minmax,atraso_entrega_dias_zscore,review_score_minmax,review_score_zscore
0,00fc308dd7f0937682698becaa9dcc45,1,53e4c6e0f4312d4d2107a8c9cddf45cd,940d38f0ef286bd93612fcdc91dd38ed,utilidades_domesticas,delivered,2018-08-20 19:51:32,85.71,-1.239028,0.0,...,1,0,0,0,0.372364,0.530413,0.537844,0.270534,1.00,0.698370
1,0124e0c5bfae1979838a91e47df53f14,1,1f50f920176fa81dab994f9023523100,0bcc3eeca39e1064258aa1e932269894,ferramentas_jardim,delivered,2018-02-08 15:31:37,49.90,0.147650,1.0,...,1,0,0,0,0.417473,0.929885,0.517278,0.145913,0.75,-0.009567
2,042649285cd08ef7c546445419b43407,1,855668e0971d4dfd7bef1b6a4133b41b,927225847efa8853ebf52c9b5b9b8cc6,bebes,delivered,2018-03-13 15:24:32,59.99,-12.685856,0.0,...,1,0,0,0,0.000000,-2.767179,0.186860,-1.856267,0.50,-0.717504
3,0452166f3b0d8e378179ecea1efdfe05,1,ff1e15b778c700abdd4d239b81ac466d,40df4be4429553e18755913ac3cfeac5,instrumentos_musicais,delivered,2018-08-10 11:25:41,59.90,-0.848877,0.0,...,1,0,0,0,0.385056,0.642807,0.668778,1.063935,1.00,0.698370
4,0589d8b975f21fea2ad626ea99527911,1,a416b6a846a11724393025641d4edd5e,ba01cebc5a3c2494d0a97d4713ec8938,beleza_saude,delivered,2018-02-27 16:36:42,201.34,-4.905220,0.0,...,1,0,0,0,0.253103,-0.525741,0.352669,-0.851539,0.50,-0.717504


PARTE 10: AMOSTRA DA APRESENTAÇÃO

Envie os quatro CSVs para a pasta `/content` do Colab e rode a célula.

In [12]:
# [IA] registro 2
brutos_amostra = carregar_brutos("/content")
tabela_apresentacao = preprocessar(**brutos_amostra)
tabela_apresentacao.head()

Tabela final: 336 linhas x 28 colunas


,order_id,order_item_id,seller_id,product_id,product_category_name,order_status,order_purchase_timestamp,price,atraso_despacho_dias,flag_despacho_atrasado,...,porte_pequeno,porte_medio,porte_grande,flag_pedido_cancelado,atraso_despacho_dias_minmax,atraso_despacho_dias_zscore,atraso_entrega_dias_minmax,atraso_entrega_dias_zscore,review_score_minmax,review_score_zscore
0,0010b2e5201cc5f1ae7e9c6cc8f5bd00,1,3504c0cb71d7fa48d967e0e4c94d59d9,5a419dbf24a8c9718fe522b81c69f61a,cool_stuff,delivered,2017-09-11 17:39:33,48.90,-3.042824,0.0,...,1,0,0,0,0.578768,0.121103,0.502576,0.884177,1.0,0.625881
1,02d5935847b855f189ad0391968773b2,1,fa40cc5b934574b62717c68f3d678b6d,d5b703c271f43f9c588a6f512a00f77e,brinquedos,delivered,2017-10-13 12:32:41,34.90,-1.864190,0.0,...,1,0,0,0,0.621761,0.555718,0.455552,0.450763,1.0,0.625881
2,03059ebda0b2ad61612b300fe34ed8e2,1,45ba18c210d42734ec52c0c1c574e9ee,49ae25c5f82cd4bc8057d0b16d45af3a,construcao_ferramentas_construcao,delivered,2018-05-10 21:57:31,39.90,-5.475012,0.0,...,1,0,0,0,0.490048,-0.775753,0.481921,0.693805,1.0,0.625881
3,03b936f126145f2bb36600400aedb13a,1,d20b021d3efdf267a402c402a48ea64b,5aad359a63cbb4fc35ead44346a3eeb4,moveis_decoracao,delivered,2018-02-18 08:42:06,10.99,8.504965,1.0,...,1,0,0,0,1.000000,4.379290,0.667219,2.401674,0.0,-2.494184
4,045e9ebd4d41affc66333d58c3d107c1,1,a7f13822ceb966b076af67121f87b063,143bc7fcaf6929ef5979e58abf39d4ed,moveis_escritorio,delivered,2017-03-01 19:28:08,176.96,-3.214282,0.0,...,1,0,0,0,0.572514,0.057879,0.385473,-0.195152,0.0,-2.494184
